# Cross-Regional Housing Price Prediction & Generalization Study
## Phase 3 — Feature Selection Study

### Phase 3 — Feature Selection Study

Goal: Does reducing features improve performance for Linear Regression?

Three feature sets tested:
- All 174 features (baseline)
- Lasso selected: 80 features (LassoCV alpha=88.15)
- SHAP selected: top 30, 50, 80 features

Key findings:
1. Lasso dropped individual area features (Total Bsmt SF, 1st Flr SF)
   but kept engineered TotalSF — validates our feature engineering!
2. SHAP top features: TotalSF #1, Overall Qual #2, Neighborhood #3
   — confirms H1 and H4 hypotheses from EDA
3. Linear Regression performs best with all 174 features
   — not overfitting, needs all information it can get
4. Feature selection hurts Linear Regression here
   — tree models may tell a different story in Phase 4

Decision: Use all 174 features for modeling.
SHAP top 30 saved for interpretability analysis in Phase 8.

In [11]:
from sklearn.linear_model import LinearRegression

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# Load utils
with open('/content/drive/MyDrive/Housing_Project/utils.py', 'r') as f:
    exec(f.read())

ames, kc = load_raw()
print(ames.shape, kc.shape)

Mounted at /content/drive
(2930, 82) (21613, 21)


### 3.1 Loading processed data from phase : 2

In [2]:
X_train = load_processed('X_train.csv')
X_test = load_processed('X_test.csv')
y_train = load_processed('y_train.csv').squeeze()
y_test = load_processed('y_test.csv').squeeze()

print(X_train.shape, X_test.shape)

(2344, 174) (586, 174)


### 3.2 Baseline — All 174 Features

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

reg = LinearRegression()

reg.fit(X_train, y_train)

rmse_value = root_mean_squared_error(y_test, reg.predict(X_test))
print(rmse_value)

31267.734257005126


### 3.3 Lasso Feature Selection

In [4]:
from sklearn.linear_model import LassoCV
from sklearn.metrics import root_mean_squared_error

lasso_cv = LassoCV(cv=5, random_state=42)
lasso_cv.fit(X_train, y_train)

print(f"Best alpha: {lasso_cv.alpha_}")

rmse_lasso = root_mean_squared_error(y_test, lasso_cv.predict(X_test))
print(f"Lasso RMSE: {rmse_lasso}")

Best alpha: 88.15380579937799
Lasso RMSE: 31303.71556015735


In [5]:
# Features with non-zero coefficients
selected_features = X_train.columns[lasso_cv.coef_ != 0]
print(f"Features selected by Lasso: {len(selected_features)} out of {X_train.shape[1]}")
print(selected_features.tolist())

Features selected by Lasso: 80 out of 174
['PID', 'Lot Frontage', 'Lot Area', 'Lot Shape', 'Land Slope', 'Neighborhood', 'Overall Qual', 'Overall Cond', 'Year Remod/Add', 'Exterior 1st', 'Mas Vnr Area', 'Exter Qual', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Heating QC', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Fireplaces', 'Garage Finish', 'Garage Cars', 'Garage Area', 'Garage Qual', 'Garage Cond', 'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold', 'HasFence', 'HasAlley', 'TotalSF', 'HouseAge', 'IsRemodeled', 'TotalBath', 'MS Zoning_RL', 'Land Contour_HLS', 'Land Contour_Lvl', 'Lot Config_CulDSac', 'Condition 1_Feedr', 'Condition 1_Norm', 'Condition 1_PosN', 'Condition 2_Norm', 'Bldg Type_Twnhs', 'Bldg Type_TwnhsE', 'House Style_1Story'

In [6]:
lr_lasso = LinearRegression()
lr_lasso.fit(X_train[selected_features], y_train)

y_pred_lasso = lr_lasso.predict(X_test[selected_features])
rmse_lr_lasso = root_mean_squared_error(y_test, y_pred_lasso)
print(f"Linear Regression on Lasso features RMSE: {rmse_lr_lasso}")

Linear Regression on Lasso features RMSE: 31487.381813994012


### Finding:
Lasso reduced features from 174 → 80 but RMSE slightly increased ($31,267 → $31,487). Linear Regression is not significantly overfitting with all features on this dataset.

In [7]:
dropped_features = X_train.columns[lasso_cv.coef_ == 0]
print(f"Features dropped by Lasso: {len(dropped_features)}")
print(dropped_features.tolist())

Features dropped by Lasso: 94
['Order', 'Year Built', 'Exterior 2nd', 'Exter Cond', 'BsmtFin Type 2', 'Bsmt Unf SF', 'Total Bsmt SF', '1st Flr SF', 'Full Bath', 'Garage Yr Blt', 'HasPool', 'MS Zoning_C (all)', 'MS Zoning_FV', 'MS Zoning_I (all)', 'MS Zoning_RH', 'MS Zoning_RM', 'Street_Pave', 'Land Contour_Low', 'Utilities_NoSeWa', 'Utilities_NoSewr', 'Lot Config_FR2', 'Lot Config_FR3', 'Lot Config_Inside', 'Condition 1_PosA', 'Condition 1_RRAe', 'Condition 1_RRAn', 'Condition 1_RRNe', 'Condition 1_RRNn', 'Condition 2_Feedr', 'Condition 2_PosA', 'Condition 2_PosN', 'Condition 2_RRAe', 'Condition 2_RRAn', 'Condition 2_RRNn', 'Bldg Type_2fmCon', 'Bldg Type_Duplex', 'House Style_1.5Unf', 'House Style_2.5Fin', 'House Style_2.5Unf', 'House Style_SFoyer', 'Roof Style_Gable', 'Roof Style_Gambrel', 'Roof Style_Mansard', 'Roof Style_Shed', 'Roof Matl_Membran', 'Roof Matl_Metal', 'Roof Matl_Roll', 'Roof Matl_Tar&Grv', 'Roof Matl_WdShake', 'Foundation_CBlock', 'Foundation_Slab', 'Foundation_Stone

### Finding:
Lasso dropped individual area components (Total Bsmt SF, 1st Flr SF, Full Bath) but kept engineered features (TotalSF, TotalBath) — confirms our feature engineering captured the right signals.

### 3.4 SHAP Feature Selection

In [8]:
import xgboost as xgb
import shap

xgb_model = xgb.XGBRegressor(random_state=42)
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [9]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train)

# Mean absolute SHAP value per feature
shap_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False)

print(shap_importance.head(20))

           feature    importance
57         TotalSF  23577.953125
7     Overall Qual  20184.453125
6     Neighborhood   6516.854980
60       TotalBath   4123.060059
36    Kitchen Qual   3522.031982
3         Lot Area   3140.348633
10  Year Remod/Add   2869.780762
8     Overall Cond   2511.591064
29     Gr Liv Area   2390.907471
20    BsmtFin SF 1   1893.401611
38      Fireplaces   1819.867798
19  BsmtFin Type 1   1556.774536
18   Bsmt Exposure   1501.283569
23     Bsmt Unf SF   1442.308716
27      2nd Flr SF   1394.706177
58        HouseAge   1362.474731
1              PID   1208.948730
9       Year Built   1000.744019
12    Exterior 2nd    917.510132
16       Bsmt Qual    917.022583


In [10]:
# Select top 30 SHAP features
top_n = 30
shap_selected = shap_importance.head(top_n)['feature'].tolist()
print(f"SHAP selected {top_n} features")
print(shap_selected)

SHAP selected 30 features
['TotalSF', 'Overall Qual', 'Neighborhood', 'TotalBath', 'Kitchen Qual', 'Lot Area', 'Year Remod/Add', 'Overall Cond', 'Gr Liv Area', 'BsmtFin SF 1', 'Fireplaces', 'BsmtFin Type 1', 'Bsmt Exposure', 'Bsmt Unf SF', '2nd Flr SF', 'HouseAge', 'PID', 'Year Built', 'Exterior 2nd', 'Bsmt Qual', '1st Flr SF', 'Sale Condition_Normal', 'Total Bsmt SF', 'Central Air_Y', 'Garage Yr Blt', 'Wood Deck SF', 'Heating QC', 'Condition 1_Norm', 'Order', 'Screen Porch']


In [12]:
lr_shap = LinearRegression()
lr_shap.fit(X_train[shap_selected], y_train)

y_pred_shap = lr_shap.predict(X_test[shap_selected])
rmse_lr_shap = root_mean_squared_error(y_test, y_pred_shap)

print(f"Linear Regression on SHAP features RMSE: {rmse_lr_shap}")

Linear Regression on SHAP features RMSE: 33463.89545217653


In [13]:
top_n = 50
shap_selected = shap_importance.head(top_n)['feature'].tolist()

lr_shap = LinearRegression()
lr_shap.fit(X_train[shap_selected], y_train)
rmse_lr_shap = root_mean_squared_error(y_test, lr_shap.predict(X_test[shap_selected]))
print(f"SHAP top 50 RMSE: {rmse_lr_shap}")

SHAP top 50 RMSE: 32729.16627762844


In [14]:
top_n = 80
shap_selected = shap_importance.head(top_n)['feature'].tolist()

lr_shap = LinearRegression()
lr_shap.fit(X_train[shap_selected], y_train)
rmse_lr_shap = root_mean_squared_error(y_test, lr_shap.predict(X_test[shap_selected]))
print(f"SHAP top 80 RMSE: {rmse_lr_shap}")

SHAP top 80 RMSE: 31569.079743829498


### Finding:

We tried reducing features from 174 to fewer using Lasso and SHAP, hoping it would reduce overfitting and improve test performance. But Linear Regression actually performed best with all 174 features. This tells us Linear Regression is not overfitting badly on this dataset — it needs all the information it can get. Tree-based models like Random Forest and XGBoost might behave differently since they handle high dimensions better.

### 3.5 Comparison Table

In [15]:
results = {
    'Feature Set': ['All 174', 'Lasso (80)', 'SHAP top 30', 'SHAP top 50', 'SHAP top 80'],
    'Features': [174, 80, 30, 50, 80],
    'RMSE': [31267, 31487, 33463, 32729, 31569]
}

pd.DataFrame(results).sort_values('RMSE')

,Feature Set,Features,RMSE
0,All 174,174,31267
1,Lasso (80),80,31487
4,SHAP top 80,80,31569
3,SHAP top 50,50,32729
2,SHAP top 30,30,33463


In [16]:
save_processed(pd.DataFrame(shap_importance), 'shap_importance.csv')
print("Saved!")

Saved: shap_importance.csv
Saved!
